# Libraries

In [ ]:
import pickle
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from pysr import PySRRegressor
import matplotlib.pyplot as plt
import itertools

# Read dataset

In [ ]:
filename = 'dataset_1.pkl'
with open(filename, 'rb') as f:
    dataset = pickle.load(f)

In [ ]:
x = dataset[1]
x.head()

In [ ]:
y = dataset[2]
y.head()

# Symbolic Regression

### Grid Search parameters

In [ ]:
n_folds = 10                                # Number of folds for cross-validation
size_test = 0.1                             # Proportion of the dataset to include in the test split
parsimony_levels = [0.001, 0.01]            # Regularization levels to control model complexity
max_size_levels = [20, 30]                  # Maximum size of the equations
pop_size_levels = [250, 500]                # Population size for the evolutionary algorithm
ncycles_per_iteration_levels = [1000]       # Number of cycles per iteration
niterations_levels = [500]                  # Number of iterations
weight_optimize_levels = [0.001]            # Weight optimization levels

In [ ]:
combs_ = list(itertools.product(parsimony_levels, max_size_levels, pop_size_levels, ncycles_per_iteration_levels, niterations_levels, weight_optimize_levels))
da = pd.DataFrame(combs_, columns=['parsimony', 'max_size', 'pop_size', 'ncycles_per_iteration', 'niterations', 'weight_optimize'])
da

### Train and Test

In [ ]:
dx = []
for j in range(n_folds):
   # Split the data in training and testing sets
   x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=size_test)
   # Standardize the features using StandardScaler z-score normalization
   scaler = StandardScaler()
   scaler.fit(x_train)
   x_train_scaled = scaler.transform(x_train)
   x_train_scaled = pd.DataFrame(x_train_scaled, columns=x.columns)
   x_test_scaled = scaler.transform(x_test)
   x_test_scaled = pd.DataFrame(x_test_scaled, columns=x.columns)
   
   for i, row in da.iterrows():
      # Save the scaler parameters
      dict_res = {}
      for i, col_name in enumerate(x_train.columns):
         dict_res[col_name] = {"avg": scaler.mean_[i], "std_dev": scaler.scale_[i]}
      
      # Train the PySR model
      model = PySRRegressor(
                              parsimony=row['parsimony'],           
                              maxsize=int(row['max_size']),
                              maxdepth=30,
                              population_size=int(row['pop_size']),
                              ncycles_per_iteration=int(row['ncycles_per_iteration']),
                              niterations=int(row['niterations']), 
                              adaptive_parsimony_scaling=50,
                              deterministic=False, 
                              should_optimize_constants=True,
                              optimizer_algorithm="BFGS",
                              optimizer_nrestarts=3,
                              binary_operators=["+", "-", "*", "/"],
                              unary_operators=["sin", "cos", "exp", "sqrt", "log", "abs", "square", "cube"],
                              weight_optimize=row['weight_optimize'],
                              verbosity=0,
                              progress=False,
                              parallelism='multithreading',
                              early_stop_condition=1e-8,
                           )
      model.fit(x_train_scaled, y_train)
      
      # Statistics and results
      y_pred_train = model.predict(x_train_scaled)
      y_pred_test = model.predict(x_test_scaled)
      rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
      rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
      r2_train = r2_score(y_train, y_pred_train)
      r2_test = r2_score(y_test, y_pred_test)
      print(f"Train RMSE: {rmse_train}")
      print(f"Test RMSE: {rmse_test}")
      print(f"Train R2: {r2_train}")
      print(f"Test R2: {r2_test}")
      best_eq = model.get_best()
      best_equation = best_eq['equation']
      print(f"Best Equation: {best_equation}")
      print("\n\n")
      dict_res['model'] = {
                              "train_rmse": rmse_train,
                              "test_rmse": rmse_test,
                              "train_r2": r2_train,
                              "test_r2": r2_test,
                              "best_equation": best_equation,
                              "scaler_params": scaler,
                              "model_object": model
                          }
      dx.append(dict_res)

### Save data

In [ ]:
aux = [{'metadados': 'index [1] data of model trained fold=0 grid=0, index [2] data of model trained fold=0 grid=1, ...'}]
results = aux + dx
with open(f'results_pysr_model_9010_{filename}.pkl', 'wb') as f:
    pickle.dump(results, f)